# Production-Grade Portfolio Optimisation v2

This notebook is a stronger, more practical version of the original framework.

It is built for:
- UK equities by default
- dividend-aware portfolios
- benchmark-relative risk control
- realistic concentration limits
- walk-forward validation

## Main upgrades
- robust return and covariance estimation
- benchmark-aware optimisation with active-weight controls
- sector caps and optional sector floors
- dividend-yield tilt and optional minimum portfolio yield
- tracking-error penalty and optional tracking-error ceiling
- turnover penalty and transaction-cost drag
- efficient frontier and active-risk diagnostics
- walk-forward backtest


In [ ]:
# If needed, uncomment:
# %pip install yfinance scikit-learn scipy matplotlib pandas numpy

from __future__ import annotations

import warnings
warnings.filterwarnings("ignore")

from dataclasses import dataclass, field
from typing import Dict, Iterable, Optional, Tuple, List

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import minimize
from sklearn.covariance import LedoitWolf
import yfinance as yf


## 1. Configuration

Edit this section for most practical use cases.

Notes:
- Use Yahoo Finance tickers with `.L` for London names.
- `benchmark_weights` should refer to the same investable universe as `tickers`.
- Dividend yields can be replaced with your own broker or vendor numbers.


In [ ]:
@dataclass
class PortfolioConfig:
    tickers: List[str] = field(default_factory=lambda: [
        "SHEL.L", "AZN.L", "BP.L", "GSK.L", "ULVR.L", "NG.L",
        "REL.L", "RIO.L", "LSEG.L", "BATS.L", "TSCO.L", "HSBA.L"
    ])
    benchmark: str = "^FTSE"
    start_date: str = "2016-01-01"
    end_date: Optional[str] = None

    frequency: str = "M"
    risk_free_rate_annual: float = 0.04

    objective: str = "max_sharpe"   # min_variance | max_sharpe | min_cvar
    target_return_annual: Optional[float] = None

    min_weight: float = 0.0
    max_weight: float = 0.15
    allow_short: bool = False

    mean_shrink: float = 0.60
    l2_reg: float = 0.02
    turnover_penalty: float = 0.002
    transaction_cost_bps: float = 10.0

    cvar_alpha: float = 0.95

    benchmark_weights: Dict[str, float] = field(default_factory=lambda: {
        "SHEL.L": 0.12, "AZN.L": 0.14, "BP.L": 0.08, "GSK.L": 0.07,
        "ULVR.L": 0.06, "NG.L": 0.05, "REL.L": 0.06, "RIO.L": 0.08,
        "LSEG.L": 0.07, "BATS.L": 0.06, "TSCO.L": 0.05, "HSBA.L": 0.16
    })
    active_weight_limit: Optional[float] = 0.05
    tracking_error_penalty: float = 0.20
    max_tracking_error_annual: Optional[float] = 0.12

    sector_map: Dict[str, str] = field(default_factory=lambda: {
        "SHEL.L": "Energy", "BP.L": "Energy",
        "AZN.L": "Healthcare", "GSK.L": "Healthcare",
        "ULVR.L": "Consumer Staples", "TSCO.L": "Consumer Staples", "BATS.L": "Consumer Staples",
        "NG.L": "Utilities",
        "REL.L": "Information Services",
        "LSEG.L": "Financials", "HSBA.L": "Financials",
        "RIO.L": "Materials",
    })
    sector_caps: Dict[str, float] = field(default_factory=lambda: {
        "Energy": 0.20,
        "Healthcare": 0.20,
        "Consumer Staples": 0.22,
        "Utilities": 0.10,
        "Financials": 0.22,
        "Materials": 0.12,
        "Information Services": 0.12,
    })
    sector_mins: Dict[str, float] = field(default_factory=dict)

    dividend_yields_annual: Dict[str, float] = field(default_factory=lambda: {
        "SHEL.L": 0.039, "AZN.L": 0.021, "BP.L": 0.050, "GSK.L": 0.036,
        "ULVR.L": 0.033, "NG.L": 0.056, "REL.L": 0.012, "RIO.L": 0.059,
        "LSEG.L": 0.011, "BATS.L": 0.079, "TSCO.L": 0.035, "HSBA.L": 0.067
    })
    dividend_tilt_strength: float = 0.10
    min_portfolio_dividend_yield_annual: Optional[float] = None

    walk_forward_train_months: int = 36
    walk_forward_test_months: int = 3
    rebalance_every_months: int = 3

config = PortfolioConfig()
config


## 2. Data download and cleaning

In [ ]:
def download_adjusted_prices(
    tickers: Iterable[str],
    start_date: str,
    end_date: Optional[str] = None,
) -> pd.DataFrame:
    tickers = list(dict.fromkeys(tickers))
    data = yf.download(
        tickers=tickers,
        start=start_date,
        end=end_date,
        auto_adjust=True,
        progress=False,
        group_by="ticker",
        threads=True,
    )

    if data.empty:
        raise ValueError("No data returned. Check tickers or date range.")

    if len(tickers) == 1:
        px = data[["Close"]].rename(columns={"Close": tickers[0]})
    else:
        close_cols = {}
        for t in tickers:
            if (t, "Close") in data.columns:
                close_cols[(t, "Close")] = t
        px = data.loc[:, list(close_cols.keys())].copy()
        px.columns = [close_cols[c] for c in px.columns]

    px = px.sort_index().ffill().dropna(how="all")
    return px

def resample_returns(prices: pd.DataFrame, frequency: str = "M") -> pd.DataFrame:
    sampled = prices.resample(frequency).last()
    returns = sampled.pct_change().dropna(how="all")
    returns = returns.replace([np.inf, -np.inf], np.nan).dropna(axis=1, how="any")
    return returns

asset_prices = download_adjusted_prices(config.tickers, config.start_date, config.end_date)
benchmark_prices = download_adjusted_prices([config.benchmark], config.start_date, config.end_date)

returns = resample_returns(asset_prices, config.frequency)
benchmark_returns = resample_returns(benchmark_prices, config.frequency)

common = returns.columns.tolist()
returns.head()


## 3. Helpers for benchmark, sector, and dividend controls

In [ ]:
def annualise_return(monthly_return: float) -> float:
    return (1 + monthly_return) ** 12 - 1

def deannualise_return(annual_return: float) -> float:
    return (1 + annual_return) ** (1 / 12) - 1

def annual_to_monthly_rate(x: float) -> float:
    return (1 + x) ** (1 / 12) - 1

def normalise_weights_dict(weights: Dict[str, float], tickers: List[str]) -> pd.Series:
    s = pd.Series(weights, dtype=float).reindex(tickers).fillna(0.0)
    total = float(s.sum())
    if total <= 0:
        return pd.Series(1.0 / len(tickers), index=tickers)
    return s / total

def get_benchmark_weights(config: PortfolioConfig, tickers: List[str]) -> pd.Series:
    return normalise_weights_dict(config.benchmark_weights, tickers)

def get_dividend_yields(config: PortfolioConfig, tickers: List[str]) -> pd.Series:
    return pd.Series(config.dividend_yields_annual, dtype=float).reindex(tickers).fillna(0.0)

def sector_exposures(weights: pd.Series, sector_map: Dict[str, str]) -> pd.Series:
    df = pd.DataFrame({"weight": weights})
    df["sector"] = [sector_map.get(t, "Unclassified") for t in df.index]
    return df.groupby("sector")["weight"].sum().sort_values(ascending=False)

def portfolio_dividend_yield_annual(weights: np.ndarray, dividend_yields_annual: pd.Series) -> float:
    return float(np.dot(weights, dividend_yields_annual.values))

def estimate_expected_returns(returns: pd.DataFrame, shrink: float = 0.60) -> pd.Series:
    sample_mu = returns.mean()
    grand_mean = float(sample_mu.mean())
    shrunk_mu = (1 - shrink) * sample_mu + shrink * grand_mean
    return shrunk_mu

def estimate_covariance(returns: pd.DataFrame) -> pd.DataFrame:
    model = LedoitWolf().fit(returns.values)
    cov = pd.DataFrame(model.covariance_, index=returns.columns, columns=returns.columns)
    return cov


## 4. Optimiser

The optimiser supports:
- max Sharpe
- min variance
- min CVaR

It also includes:
- active-weight limits vs benchmark
- tracking-error penalty and optional ceiling
- sector caps and floors
- turnover penalty
- dividend tilt


In [ ]:
def portfolio_return(weights: np.ndarray, mu: pd.Series) -> float:
    return float(np.dot(weights, mu.values))

def portfolio_variance(weights: np.ndarray, cov: pd.DataFrame) -> float:
    return float(weights @ cov.values @ weights)

def portfolio_volatility(weights: np.ndarray, cov: pd.DataFrame) -> float:
    return float(np.sqrt(max(portfolio_variance(weights, cov), 0.0)))

def portfolio_cvar(weights: np.ndarray, scenario_returns: pd.DataFrame, alpha: float = 0.95) -> float:
    portfolio_path = scenario_returns.values @ weights
    losses = -portfolio_path
    threshold = np.quantile(losses, alpha)
    tail = losses[losses >= threshold]
    if len(tail) == 0:
        return float(max(0.0, losses.max()))
    return float(np.mean(tail))

def tracking_error_monthly(weights: np.ndarray, benchmark_weights: np.ndarray, cov: pd.DataFrame) -> float:
    active = weights - benchmark_weights
    te_var = float(active @ cov.values @ active)
    return float(np.sqrt(max(te_var, 0.0)))

def build_constraints(
    mu: pd.Series,
    cov: pd.DataFrame,
    config: PortfolioConfig,
    benchmark_weights: pd.Series,
    dividend_yields_annual: pd.Series,
) -> List[dict]:
    constraints = [{"type": "eq", "fun": lambda w: np.sum(w) - 1.0}]

    if config.target_return_annual is not None:
        target_m = deannualise_return(config.target_return_annual)
        constraints.append({"type": "ineq", "fun": lambda w, target_m=target_m: portfolio_return(w, mu) - target_m})

    if config.max_tracking_error_annual is not None:
        te_limit_m = config.max_tracking_error_annual / np.sqrt(12)
        bmk = benchmark_weights.values.copy()
        constraints.append({
            "type": "ineq",
            "fun": lambda w, te_limit_m=te_limit_m, bmk=bmk: te_limit_m - tracking_error_monthly(w, bmk, cov)
        })

    if config.min_portfolio_dividend_yield_annual is not None:
        min_div = config.min_portfolio_dividend_yield_annual
        dy = dividend_yields_annual.values.copy()
        constraints.append({
            "type": "ineq",
            "fun": lambda w, min_div=min_div, dy=dy: np.dot(w, dy) - min_div
        })

    sectors = sorted(set(config.sector_map.get(t, "Unclassified") for t in mu.index))
    for sector in sectors:
        names = [t for t in mu.index if config.sector_map.get(t, "Unclassified") == sector]
        idx = [mu.index.get_loc(t) for t in names]
        cap = config.sector_caps.get(sector)
        floor = config.sector_mins.get(sector)
        if cap is not None:
            constraints.append({"type": "ineq", "fun": lambda w, idx=idx, cap=cap: cap - np.sum(w[idx])})
        if floor is not None:
            constraints.append({"type": "ineq", "fun": lambda w, idx=idx, floor=floor: np.sum(w[idx]) - floor})

    return constraints

def optimise_portfolio(
    returns: pd.DataFrame,
    mu: pd.Series,
    cov: pd.DataFrame,
    config: PortfolioConfig,
    previous_weights: Optional[pd.Series] = None,
) -> pd.Series:
    tickers = list(mu.index)
    n = len(tickers)

    benchmark_weights = get_benchmark_weights(config, tickers)
    dividend_yields = get_dividend_yields(config, tickers)

    if previous_weights is None:
        previous_weights = benchmark_weights.copy()
    else:
        previous_weights = previous_weights.reindex(tickers).fillna(0.0)
        total = float(previous_weights.sum())
        previous_weights = previous_weights / total if total > 0 else benchmark_weights.copy()

    if config.allow_short:
        bounds = [(-config.max_weight, config.max_weight) for _ in range(n)]
    else:
        bounds = [(config.min_weight, config.max_weight) for _ in range(n)]

    if config.active_weight_limit is not None:
        bounds = [
            (max(b[0], benchmark_weights.iloc[i] - config.active_weight_limit),
             min(b[1], benchmark_weights.iloc[i] + config.active_weight_limit))
            for i, b in enumerate(bounds)
        ]

    constraints = build_constraints(mu, cov, config, benchmark_weights, dividend_yields)

    rf_m = annual_to_monthly_rate(config.risk_free_rate_annual)
    prev = previous_weights.values.copy()
    bmk = benchmark_weights.values.copy()
    dy = dividend_yields.values.copy()

    def objective(w: np.ndarray) -> float:
        ret = portfolio_return(w, mu)
        vol = portfolio_volatility(w, cov)
        cvar = portfolio_cvar(w, returns, alpha=config.cvar_alpha)
        turnover = np.sum(np.abs(w - prev))
        l2 = np.sum(w ** 2)
        te = tracking_error_monthly(w, bmk, cov)
        dividend_bonus = np.dot(w, dy)

        penalty = (
            config.turnover_penalty * turnover
            + config.l2_reg * l2
            + config.tracking_error_penalty * te
            - config.dividend_tilt_strength * dividend_bonus
        )

        if config.objective == "min_variance":
            return vol + penalty
        if config.objective == "min_cvar":
            return cvar + penalty
        if config.objective == "max_sharpe":
            sharpe = (ret - rf_m) / max(vol, 1e-10)
            return -sharpe + penalty
        raise ValueError(f"Unknown objective: {config.objective}")

    x0 = benchmark_weights.values.copy()
    result = minimize(
        objective,
        x0=x0,
        method="SLSQP",
        bounds=bounds,
        constraints=constraints,
        options={"maxiter": 500, "ftol": 1e-9, "disp": False},
    )

    if not result.success:
        raise RuntimeError(f"Optimisation failed: {result.message}")

    weights = pd.Series(result.x, index=tickers, name="weight")
    if not config.allow_short:
        weights = weights.clip(lower=0.0)
    weights = weights / weights.sum()
    return weights


## 5. Run the optimiser on the full sample

In [ ]:
mu = estimate_expected_returns(returns, shrink=config.mean_shrink)
cov = estimate_covariance(returns)

weights = optimise_portfolio(
    returns=returns,
    mu=mu,
    cov=cov,
    config=config,
    previous_weights=None,
)

weights.sort_values(ascending=False)


## 6. Diagnostics and portfolio summary

In [ ]:
def risk_contributions(weights: pd.Series, cov: pd.DataFrame) -> pd.Series:
    w = weights.values
    port_vol = portfolio_volatility(w, cov)
    if port_vol <= 0:
        return pd.Series(0.0, index=weights.index)
    marginal = cov.values @ w / port_vol
    contrib = w * marginal
    contrib_pct = contrib / contrib.sum()
    return pd.Series(contrib_pct, index=weights.index)

def build_portfolio_summary(
    weights: pd.Series,
    returns: pd.DataFrame,
    mu: pd.Series,
    cov: pd.DataFrame,
    risk_free_rate_annual: float,
    benchmark_weights: Optional[pd.Series] = None,
    sector_map: Optional[Dict[str, str]] = None,
    dividend_yields_annual: Optional[pd.Series] = None,
    alpha: float = 0.95,
) -> Tuple[pd.DataFrame, Dict[str, float], Optional[pd.DataFrame]]:
    w = weights.values
    port_ret_m = portfolio_return(w, mu)
    port_vol_m = portfolio_volatility(w, cov)
    rf_m = annual_to_monthly_rate(risk_free_rate_annual)

    asset_table = pd.DataFrame({
        "weight": weights,
        "expected_return_annual": mu.apply(annualise_return),
        "volatility_annual": np.sqrt(np.diag(cov.values)) * np.sqrt(12),
        "risk_contribution_pct": risk_contributions(weights, cov),
    })

    metrics = {
        "expected_return_annual": annualise_return(port_ret_m),
        "volatility_annual": port_vol_m * np.sqrt(12),
        "sharpe_ratio": (port_ret_m - rf_m) / max(port_vol_m, 1e-10),
        "cvar_monthly": portfolio_cvar(w, returns, alpha=alpha),
    }

    sector_table = None

    if benchmark_weights is not None:
        benchmark_weights = benchmark_weights.reindex(weights.index).fillna(0.0)
        asset_table["benchmark_weight"] = benchmark_weights
        asset_table["active_weight"] = asset_table["weight"] - asset_table["benchmark_weight"]
        metrics["tracking_error_annual"] = tracking_error_monthly(
            weights.values, benchmark_weights.values, cov
        ) * np.sqrt(12)

    if dividend_yields_annual is not None:
        dividend_yields_annual = dividend_yields_annual.reindex(weights.index).fillna(0.0)
        asset_table["dividend_yield_annual"] = dividend_yields_annual
        metrics["portfolio_dividend_yield_annual"] = portfolio_dividend_yield_annual(
            weights.values, dividend_yields_annual
        )

    if sector_map is not None:
        sector_table = sector_exposures(weights, sector_map).rename("portfolio_weight").to_frame()
        if benchmark_weights is not None:
            sector_table["benchmark_weight"] = sector_exposures(benchmark_weights, sector_map)
            sector_table["active_weight"] = sector_table["portfolio_weight"] - sector_table["benchmark_weight"]
        sector_table = sector_table.fillna(0.0)

    return asset_table.sort_values("weight", ascending=False), metrics, sector_table

benchmark_weights = get_benchmark_weights(config, list(weights.index))
dividend_yields = get_dividend_yields(config, list(weights.index))

asset_table, stats, sector_table = build_portfolio_summary(
    weights=weights,
    returns=returns,
    mu=mu,
    cov=cov,
    risk_free_rate_annual=config.risk_free_rate_annual,
    benchmark_weights=benchmark_weights,
    sector_map=config.sector_map,
    dividend_yields_annual=dividend_yields,
    alpha=config.cvar_alpha,
)

display(asset_table.round(4))
display(pd.Series(stats).round(4))
if sector_table is not None:
    display(sector_table.round(4))


## 7. Efficient frontier under the same robust estimators

In [ ]:
def efficient_frontier(
    returns: pd.DataFrame,
    config: PortfolioConfig,
    n_points: int = 20,
) -> pd.DataFrame:
    mu = estimate_expected_returns(returns, shrink=config.mean_shrink)
    cov = estimate_covariance(returns)

    min_ret = annualise_return(mu.min())
    max_ret = annualise_return(mu.max())
    target_returns = np.linspace(min_ret, max_ret, n_points)

    rows = []
    base_target = config.target_return_annual
    base_objective = config.objective

    for target in target_returns:
        trial = PortfolioConfig(**{**config.__dict__})
        trial.objective = "min_variance"
        trial.target_return_annual = float(target)

        try:
            w = optimise_portfolio(returns, mu, cov, trial, previous_weights=None)
            rows.append({
                "target_return_annual": target,
                "volatility_annual": portfolio_volatility(w.values, cov) * np.sqrt(12),
                "dividend_yield_annual": portfolio_dividend_yield_annual(
                    w.values, get_dividend_yields(config, list(mu.index))
                ),
            })
        except Exception:
            continue

    config.target_return_annual = base_target
    config.objective = base_objective
    return pd.DataFrame(rows)

frontier = efficient_frontier(returns, config, n_points=18)

plt.figure(figsize=(8, 5))
plt.plot(frontier["volatility_annual"] * 100, frontier["target_return_annual"] * 100, marker="o")
plt.xlabel("Annualised volatility (%)")
plt.ylabel("Target annual return (%)")
plt.title("Efficient Frontier")
plt.grid(alpha=0.3)
plt.show()


## 8. Weight, active-weight, and sector charts

In [ ]:
plot_table = asset_table.copy()

plt.figure(figsize=(9, 4))
plt.bar(plot_table.index, plot_table["weight"] * 100)
plt.xticks(rotation=45, ha="right")
plt.ylabel("Weight (%)")
plt.title("Portfolio Weights")
plt.grid(axis="y", alpha=0.3)
plt.show()

if "active_weight" in plot_table.columns:
    plt.figure(figsize=(9, 4))
    plt.bar(plot_table.index, plot_table["active_weight"] * 100)
    plt.xticks(rotation=45, ha="right")
    plt.ylabel("Active weight (%)")
    plt.title("Active Weights vs Benchmark")
    plt.grid(axis="y", alpha=0.3)
    plt.show()

if sector_table is not None:
    plt.figure(figsize=(8, 4))
    plt.bar(sector_table.index, sector_table["portfolio_weight"] * 100)
    plt.xticks(rotation=30, ha="right")
    plt.ylabel("Sector weight (%)")
    plt.title("Sector Exposure")
    plt.grid(axis="y", alpha=0.3)
    plt.show()


## 9. Walk-forward backtest

In [ ]:
def walk_forward_backtest(
    returns: pd.DataFrame,
    config: PortfolioConfig,
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    rows = []
    weights_history = []
    prev_weights = None

    train = config.walk_forward_train_months
    test = config.walk_forward_test_months
    step = config.rebalance_every_months

    for start in range(0, len(returns) - train - test + 1, step):
        train_slice = returns.iloc[start : start + train]
        test_slice = returns.iloc[start + train : start + train + test]

        mu = estimate_expected_returns(train_slice, config.mean_shrink)
        cov = estimate_covariance(train_slice)

        try:
            w = optimise_portfolio(
                returns=train_slice,
                mu=mu,
                cov=cov,
                config=config,
                previous_weights=prev_weights,
            )
        except Exception:
            continue

        if prev_weights is None:
            turnover = float(w.abs().sum())
        else:
            aligned_prev = prev_weights.reindex(w.index).fillna(0.0)
            turnover = float((w - aligned_prev).abs().sum())

        cost = turnover * config.transaction_cost_bps / 10000.0

        for dt, row in test_slice.iterrows():
            gross = float(np.dot(w.values, row.reindex(w.index).values))
            net = gross if dt != test_slice.index[0] else gross - cost
            rows.append({
                "date": dt,
                "portfolio_return": net,
                "gross_return": gross,
                "cost": cost if dt == test_slice.index[0] else 0.0,
                "turnover": turnover if dt == test_slice.index[0] else 0.0,
            })

        weights_history.append(pd.DataFrame({
            "date": test_slice.index[0],
            "ticker": w.index,
            "weight": w.values,
        }))
        prev_weights = w.copy()

    perf = pd.DataFrame(rows).set_index("date").sort_index()
    if not perf.empty:
        perf["equity_curve"] = (1 + perf["portfolio_return"]).cumprod()

    wh = pd.concat(weights_history, ignore_index=True) if weights_history else pd.DataFrame()
    return perf, wh

def compute_backtest_stats(perf: pd.DataFrame, benchmark_returns: Optional[pd.DataFrame] = None) -> pd.Series:
    if perf.empty:
        return pd.Series(dtype=float)

    r = perf["portfolio_return"]
    ann_ret = (1 + r.mean()) ** 12 - 1
    ann_vol = r.std(ddof=1) * np.sqrt(12)
    sharpe = np.nan if ann_vol == 0 else (ann_ret - config.risk_free_rate_annual) / ann_vol

    cum = (1 + r).cumprod()
    running_max = cum.cummax()
    drawdown = cum / running_max - 1

    stats = {
        "Backtest annual return %": ann_ret * 100,
        "Backtest annual vol %": ann_vol * 100,
        "Backtest Sharpe": sharpe,
        "Max drawdown %": drawdown.min() * 100,
        "Average turnover %": perf["turnover"].mean() * 100,
        "Average cost bps": perf["cost"].mean() * 10000,
    }

    if benchmark_returns is not None and not benchmark_returns.empty:
        bench = benchmark_returns.reindex(perf.index).iloc[:, 0].dropna()
        aligned = perf["portfolio_return"].reindex(bench.index).dropna()
        bench = bench.reindex(aligned.index)
        if len(aligned) > 1:
            active = aligned - bench
            te = active.std(ddof=1) * np.sqrt(12)
            stats["Backtest tracking error %"] = te * 100
            stats["Backtest information ratio"] = np.nan if te == 0 else (active.mean() * 12) / te

    return pd.Series(stats)

perf, weights_history = walk_forward_backtest(returns, config)
bt_stats = compute_backtest_stats(perf, benchmark_returns)

display(bt_stats.round(3))

if not perf.empty:
    plt.figure(figsize=(9, 5))
    plt.plot(perf.index, perf["equity_curve"], label="Strategy")
    plt.title("Walk-Forward Equity Curve")
    plt.ylabel("Growth of £1")
    plt.grid(alpha=0.3)
    plt.legend()
    plt.show()

    plt.figure(figsize=(9, 4))
    plt.bar(perf.index, perf["turnover"] * 100)
    plt.title("Turnover by Rebalance")
    plt.ylabel("Turnover (%)")
    plt.grid(axis="y", alpha=0.3)
    plt.show()
else:
    print("Backtest produced no observations. Try a longer history or looser constraints.")


## 10. Clean export view

In [ ]:
final_output = asset_table.copy()

rename_map = {
    "weight": "Weight %",
    "benchmark_weight": "Benchmark weight %",
    "active_weight": "Active weight %",
    "expected_return_annual": "Expected annual return %",
    "volatility_annual": "Standalone annual vol %",
    "risk_contribution_pct": "Risk contribution %",
    "dividend_yield_annual": "Dividend yield %",
}
final_output = final_output.rename(columns=rename_map)

for col in final_output.columns:
    if "%" in col:
        final_output[col] = (final_output[col] * 100).round(2)

display(final_output)
display(pd.Series(stats).mul(100).rename({
    "expected_return_annual": "Expected return %",
    "volatility_annual": "Volatility %",
    "portfolio_dividend_yield_annual": "Portfolio dividend yield %",
    "tracking_error_annual": "Tracking error %",
}).round(2))


## 11. Practical next upgrades

Natural v3 extensions:
- benchmark constituent ingestion directly from an index file
- factor exposure control such as value, quality, low vol, size
- Black-Litterman or blended views
- explicit liquidity and ADV constraints
- tax lots and realised gains
- ESG or controversy exclusions
- cardinality constraints
- scenario stress testing
- order-generation and execution hooks

For a real investment process, the important disciplines are:
1. robust data
2. realistic constraints
3. out-of-sample validation
4. stability under perturbation
5. governance over overrides
